# Project 1: Automatic Review Analyzer

This notebook applies the three linear classifiers to the **UCI Sentiment Labelled Sentences** dataset. The dataset contains 3,000 labelled sentences: 1,000 each from IMDb, Amazon, and Yelp, with 500 positive and 500 negative sentences from each source.

The reusable learning algorithms live in `linear_classification.py`. Text processing and the experimental workflow stay in this notebook.

The project story is:

**real reviews → train/validation/test → sparse feature vectors X + labels y → three linear classifiers → accuracy and training time → Pegasos lambda selection → learned word weights → two-feature classifier visualizations.**


In [ ]:
from pathlib import Path
import io
import random
import re
import sys
import time
import urllib.request
import zipfile

import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != 'project_1':
    candidate = PROJECT_DIR / 'unit_1' / 'project_1'
    if candidate.exists():
        PROJECT_DIR = candidate
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from linear_classification import accuracy, average_perceptron, pegasos, perceptron


## 1. Load the 3,000-sentence dataset

The dataset is downloaded from the UCI Machine Learning Repository when needed. It is licensed under CC BY 4.0. The source contains three files: IMDb, Amazon, and Yelp. Each row contains a sentence followed by a binary label (`1` positive, `0` negative).


In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip'
DATA_DIR = PROJECT_DIR / 'data' / 'sentiment_labelled_sentences'
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DATA_DIR / 'sentiment_labelled_sentences.zip'

if not ZIP_PATH.exists():
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(DATA_DIR)

dataset_root = next(DATA_DIR.glob('**/sentiment labelled sentences'), DATA_DIR)
source_files = {
    'IMDb': dataset_root / 'imdb_labelled.txt',
    'Amazon': dataset_root / 'amazon_cells_labelled.txt',
    'Yelp': dataset_root / 'yelp_labelled.txt',
}

def read_source(path, source):
    rows = []
    with path.open(encoding='utf-8') as handle:
        for line in handle:
            sentence, label = line.rstrip('\n').rsplit('\t', 1)
            rows.append((1 if label == '1' else -1, sentence, source))
    return rows

reviews = [row for source, path in source_files.items() for row in read_source(path, source)]
print(f'Total reviews: {len(reviews)}')
print({source: sum(1 for _, _, s in reviews if s == source) for source in source_files})
print('Positive:', sum(label == 1 for label, _, _ in reviews))
print('Negative:', sum(label == -1 for label, _, _ in reviews))


## 2. Train / validation / test split

We use a reproducible stratified split. The test set is kept untouched until the final comparison. The validation set is used for selecting Pegasos `lambda`; the test set is not used for tuning.


In [ ]:
SEED = 42
rng = random.Random(SEED)

def stratified_split(rows, train_fraction=0.70, validation_fraction=0.15):
    by_label = {1: [], -1: []}
    for row in rows:
        by_label[row[0]].append(row)
    train, validation, test = [], [], []
    for label_rows in by_label.values():
        rng.shuffle(label_rows)
        n = len(label_rows)
        train_end = int(n * train_fraction)
        validation_end = train_end + int(n * validation_fraction)
        train.extend(label_rows[:train_end])
        validation.extend(label_rows[train_end:validation_end])
        test.extend(label_rows[validation_end:])
    rng.shuffle(train)
    rng.shuffle(validation)
    rng.shuffle(test)
    return train, validation, test

train_reviews, validation_reviews, test_reviews = stratified_split(reviews)
print(f'Train: {len(train_reviews)} | Validation: {len(validation_reviews)} | Test: {len(test_reviews)}')


## 3. Convert reviews into sparse vectors

For vocabulary $V=\{w_1,\ldots,w_d\}$, each review becomes a sparse feature vector $x_i\in\mathbb{R}^d$. Labels are kept separately in $y$.

We build the vocabulary **from training data only**. Here each feature records whether a word occurs in the review.


In [ ]:
TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

def build_vocabulary(rows, min_count=2):
    counts = {}
    for _, text, _ in rows:
        for token in set(tokenize(text)):
            counts[token] = counts.get(token, 0) + 1
    words = sorted(word for word, count in counts.items() if count >= min_count)
    return {word: index for index, word in enumerate(words)}

def vectorize(rows, vocabulary):
    X, y = [], []
    for label, text, _ in rows:
        features = {}
        for token in set(tokenize(text)):
            index = vocabulary.get(token)
            if index is not None:
                features[index] = 1.0
        X.append(features)
        y.append(label)
    return X, np.asarray(y, dtype=int)

vocabulary = build_vocabulary(train_reviews)
X_train, y_train = vectorize(train_reviews, vocabulary)
X_validation, y_validation = vectorize(validation_reviews, vocabulary)
X_test, y_test = vectorize(test_reviews, vocabulary)
train_data = list(zip(y_train, X_train))
validation_data = list(zip(y_validation, X_validation))
test_data = list(zip(y_test, X_test))

print(f'Vocabulary size: {len(vocabulary)}')
print(f'X_train: {len(X_train)} sparse vectors')
print(f'y_train: {len(y_train)} labels')
print('First X vector:', X_train[0])
print('First y label:', y_train[0])


## 4. Compare the three classifiers

All three algorithms receive exactly the same training representation. We measure both validation accuracy and training time. The timings are experimental measurements, not assumptions about which algorithm must be faster.


In [ ]:
EPOCHS = 10
BATCH_SIZE = 32

def timed_train(name, train_function):
    start = time.perf_counter()
    weights = train_function()
    elapsed = time.perf_counter() - start
    return weights, elapsed

experiments = {
    'Perceptron': lambda: perceptron(train_data, epochs=EPOCHS),
    'Average Perceptron': lambda: average_perceptron(train_data, epochs=EPOCHS),
    'Pegasos': lambda: pegasos(train_data, lambda_=1e-3, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED),
}

baseline_models = {}
for name, train_function in experiments.items():
    weights, elapsed = timed_train(name, train_function)
    baseline_models[name] = weights
    print(f'{name:20} | time={elapsed:.4f}s | validation accuracy={accuracy(weights, validation_data):.4f}')


## 5. Select the best Pegasos regularization parameter

We select the smallest `lambda` that achieves the maximum validation accuracy. This is important when several values tie.

$$
\lambda^*=\arg\max_{\lambda}\mathrm{ValidationAccuracy}(\lambda).
$$


In [ ]:
lambda_grid = np.array([1e-6, 1e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1])
lambda_scores = []
lambda_times = []
for lambda_ in lambda_grid:
    start = time.perf_counter()
    weights = pegasos(train_data, lambda_=lambda_, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED)
    elapsed = time.perf_counter() - start
    score = accuracy(weights, validation_data)
    lambda_scores.append(score)
    lambda_times.append(elapsed)
    print(f'lambda={lambda_:>8.1e} | validation accuracy={score:.4f} | time={elapsed:.4f}s')

best_score = max(lambda_scores)
best_lambda = float(next(lambda_ for lambda_, score in zip(lambda_grid, lambda_scores) if np.isclose(score, best_score)))
print(f'\nSelected lambda*: {best_lambda:.1e} (smallest lambda with maximum validation accuracy)')

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.step(lambda_grid, lambda_scores, where='post', linewidth=2)
ax.scatter(lambda_grid, lambda_scores, s=55, zorder=3)
ax.scatter([best_lambda], [best_score], s=130, zorder=4, label=f'selected lambda* = {best_lambda:.1e}')
ax.axvline(best_lambda, linestyle='--', linewidth=1)
ax.set_xscale('log')
ax.set_xlabel('Pegasos regularization lambda')
ax.set_ylabel('Validation accuracy')
ax.set_title('Pegasos lambda selection on the validation set')
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()


## 6. Final comparison on the untouched test set


In [ ]:
final_models = {}
final_results = []
for name in ('Perceptron', 'Average Perceptron'):
    train_function = experiments[name]
    weights, elapsed = timed_train(name, train_function)
    final_models[name] = weights
    final_results.append((name, elapsed, accuracy(weights, validation_data), accuracy(weights, test_data)))

start = time.perf_counter()
final_models['Pegasos'] = pegasos(train_data, lambda_=best_lambda, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED)
pegasos_time = time.perf_counter() - start
final_results.append(('Pegasos', pegasos_time, accuracy(final_models['Pegasos'], validation_data), accuracy(final_models['Pegasos'], test_data)))

print(f'{"Classifier":20} | {"Time (s)":>10} | {"Validation":>10} | {"Test":>10}')
print('-' * 62)
for name, elapsed, validation_score, test_score in final_results:
    print(f'{name:20} | {elapsed:10.4f} | {validation_score:10.4f} | {test_score:10.4f}')


## 7. Learned word weights

A positive weight pushes the linear score toward the positive class; a negative weight pushes it toward the negative class. We show the strongest learned words for each classifier rather than plotting thousands of vocabulary entries.


In [ ]:
index_to_word = {index: word for word, index in vocabulary.items()}
for name, weights in final_models.items():
    ranked = sorted(((weight, index_to_word[index]) for index, weight in weights.items() if index in index_to_word), reverse=True)
    positive = ranked[:10]
    negative = sorted(ranked, key=lambda item: item[0])[:10]
    print(f'\n{name}')
    print('Positive:', [(word, round(weight, 4)) for weight, word in positive])
    print('Negative:', [(word, round(weight, 4)) for weight, word in negative])


## 8. Two-feature decision-boundary illustrations

The full classifiers above use the complete vocabulary. For a geometric illustration, we deliberately train a **separate two-feature model** using two sentiment-bearing words. This keeps the plotted boundary mathematically faithful to the two axes.

The same reviews are projected onto the two selected features, and a separate diagram is produced for each classifier. Overlapping observations are represented by their count rather than by moving their true coordinates.


In [ ]:
def select_visual_words(vocabulary, train_rows):
    positive_counts, negative_counts = {}, {}
    for label, text, _ in train_rows:
        target = positive_counts if label == 1 else negative_counts
        for token in set(tokenize(text)):
            if token in vocabulary:
                target[token] = target.get(token, 0) + 1
    positive_word = max(positive_counts, key=positive_counts.get)
    negative_word = max(negative_counts, key=negative_counts.get)
    if positive_word == negative_word:
        candidates = sorted(vocabulary, key=lambda w: positive_counts.get(w, 0) - negative_counts.get(w, 0), reverse=True)
        positive_word, negative_word = candidates[0], candidates[-1]
    return positive_word, negative_word

visual_words = select_visual_words(vocabulary, train_reviews)
print('Visualization features:', visual_words)

def two_feature_data(rows, words):
    indices = [vocabulary[words[0]], vocabulary[words[1]]]
    X, y = [], []
    for label, text, _ in rows:
        tokens = set(tokenize(text))
        X.append({0: float(words[0] in tokens), 1: float(words[1] in tokens)})
        y.append(label)
    return X, np.asarray(y)

plot_X, plot_y = two_feature_data(train_reviews, visual_words)
plot_data = list(zip(plot_y, plot_X))
plot_models = {
    'Perceptron': perceptron(plot_data, epochs=EPOCHS),
    'Average Perceptron': average_perceptron(plot_data, epochs=EPOCHS),
    'Pegasos': pegasos(plot_data, lambda_=best_lambda, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED),
}

points = np.asarray([[features[0], features[1]] for features in plot_X])
for name, weights in plot_models.items():
    fig, ax = plt.subplots(figsize=(8, 7))
    for label, marker in [(1, 'o'), (-1, 'x')]:
        mask = plot_y == label
        ax.scatter(points[mask, 0], points[mask, 1], marker=marker, s=70, alpha=0.55, label='positive' if label == 1 else 'negative')
    theta1, theta2 = weights.get(0, 0.0), weights.get(1, 0.0)
    x_values = np.linspace(-0.15, 1.15, 200)
    if abs(theta2) > 1e-12:
        ax.plot(x_values, -(theta1 / theta2) * x_values, linewidth=2.5, label='decision boundary')
    elif abs(theta1) > 1e-12:
        ax.axvline(0, linewidth=2.5, label='decision boundary')
    counts = {}
    for x1, x2 in points:
        counts[(x1, x2)] = counts.get((x1, x2), 0) + 1
    for (x1, x2), count in counts.items():
        ax.annotate(f'n={count}', (x1, x2), xytext=(7, 7), textcoords='offset points', fontsize=8)
    ax.set_xlabel(visual_words[0])
    ax.set_ylabel(visual_words[1])
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xlim(-0.15, 1.15)
    ax.set_ylim(-0.15, 1.15)
    ax.set_title(f'{name}: two-feature decision boundary')
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


## 9. Course connection

For a training example `(x, y)`, Pegasos uses the hinge-loss objective

$$
\ell(\theta;(x,y))=\max\{0,1-y\theta^Tx\}.
$$

The project therefore connects the lecture mathematics to a real text-classification experiment: the same sparse feature representation is given to all three linear learners, while Pegasos additionally controls the regularization strength with `lambda`.

Dataset source: Kotzias et al., **Sentiment Labelled Sentences**, UCI Machine Learning Repository, DOI 10.24432/C57604.
